In [1]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))

In [2]:
from src.rarekg.curated import downloaders

API_KEY = os.getenv("UMLS_API_KEY")


In [3]:
downloaders.download_umls_mrconso(API_KEY)


(PosixPath('/home/guests/andreea_magureanu/rare_disease_pipeline/data/raw/umls-2025AB-mrconso.zip'),
 PosixPath('/home/guests/andreea_magureanu/rare_disease_pipeline/data/raw/MRCONSO.RRF'))

In [ ]:


from __future__ import annotations

from pathlib import Path
from typing import Iterable, Dict, Set, List, Tuple
from collections import defaultdict
import sqlite3
from src.rarekg.utils.string_utils import normalize_name


def insert_entity(conn: sqlite3.Connection, entity_id: str, entity_type: str) -> None:
    """
    Insert an entity into the main table; if it already exists, do nothing.
    """
    conn.execute(
        "INSERT OR IGNORE INTO entity(id, type) VALUES (?, ?);",
        (entity_id, entity_type),
    )


def make_id_variant_map(ids: Iterable[str]) -> Dict[str, str]:
    """
    For IDs like 'HP:0004322' or 'HGNC:1100' or '1100', make a mapping
    from possible variants to a canonical form.

    Example:
      input: ['HP:0004322']
      variants:
        'HP:0004322' -> 'HP:0004322'
        '0004322'    -> 'HP:0004322'

      input: ['1100']
      variants:
        '1100'       -> '1100'
        'HGNC:1100'  -> 'HGNC:1100'   (we normalize genes later)
    """
    mapping: Dict[str, str] = {}
    for raw in ids:
        if raw is None:
            continue
        canonical = str(raw).strip()
        if not canonical:
            continue

        # Always map the original
        mapping[canonical] = canonical

        # If it looks like PREFIX:ID → add bare ID as variant
        if ":" in canonical:
            prefix, bare = canonical.split(":", 1)
            bare = bare.strip()
            if bare:
                mapping[bare] = canonical
        else:
            # For plain numeric IDs (like "1100"), add prefix variants that
            # might appear in UMLS (e.g., HGNC:1100, HP:0001100, etc.).
            # We only add the HGNC-style variant here; HPO we handle with full IDs.
            if canonical.isdigit():
                mapping[f"HGNC:{canonical}"] = f"HGNC:{canonical}"
    return mapping


# ---------------------------------------------------------------------------
#  PASS 1: Build resource-ID -> CUIs maps in ONE scan
# ---------------------------------------------------------------------------

def build_id_to_cui_maps(
    mrconso_path: Path,
    orpha_ids: Iterable[str],
    hpo_ids: Iterable[str],
    hgnc_ids: Iterable[str],
    rxcui_ids: Iterable[str],
) -> Tuple[
    Dict[str, Set[str]],  # OrphaCode -> CUIs
    Dict[str, Set[str]],  # HPO ID    -> CUIs
    Dict[str, Set[str]],  # HGNC ID   -> CUIs
    Dict[str, Set[str]],  # RxCUI     -> CUIs
    Set[str],             # all CUIs used
]:
    """
    Single streaming pass over MRCONSO.RRF.

    For each resource:
      - Rare diseases (Orphanet): SAB LIKE 'ORPHANET%', CODE = OrphaCode
      - Phenotypes (HPO):        SAB LIKE 'HPO%',      CODE/SCUI/SDUI = HPO ID
      - Genes (HGNC):            SAB LIKE 'HGNC%',     CODE = HGNC ID
      - Drugs (RxNorm):          SAB = 'RXNORM',       CODE = RxCUI

    Returns:
      orpha_to_cuis, hpo_to_cuis, hgnc_to_cuis, rxcui_to_cuis, all_cuis
    """

    # Normalize sets / variant maps
    orpha_set: Set[str] = {str(x).strip() for x in orpha_ids if str(x).strip()}

    hpo_variant_map = make_id_variant_map(hpo_ids)
    hgnc_variant_map = make_id_variant_map(hgnc_ids)
    rxcui_set: Set[str] = {str(x).strip() for x in rxcui_ids if str(x).strip()}

    orpha_to_cuis: Dict[str, Set[str]] = defaultdict(set)
    hpo_to_cuis: Dict[str, Set[str]] = defaultdict(set)
    hgnc_to_cuis: Dict[str, Set[str]] = defaultdict(set)
    rxcui_to_cuis: Dict[str, Set[str]] = defaultdict(set)
    all_cuis: Set[str] = set()

    with mrconso_path.open("r", encoding="utf-8") as f:
        for line in f:
            parts = line.rstrip("\n").split("|")
            if parts and parts[-1] == "":
                parts = parts[:-1]
            # Need at least up to STR (index 14)
            if len(parts) < 15:
                continue

            cui = parts[0]      # 0: CUI
            # lat = parts[1]    # 1: LAT
            scui = parts[9]     # 9: SCUI
            sdui = parts[10]    # 10: SDUI
            sab = parts[11]     # 11: SAB
            # tty = parts[12]   # 12: TTY
            code = parts[13]    # 13: CODE
            # str_val = parts[14]  # 14: STR

            # --- Orphanet: rare diseases ---
            if sab.startswith("ORPHANET") and code in orpha_set:
                orpha_to_cuis[code].add(cui)
                all_cuis.add(cui)

            # --- HPO: phenotypes ---
            if sab.startswith("HPO"):
                for candidate in (code, scui, sdui):
                    if not candidate:
                        continue
                    canonical = hpo_variant_map.get(candidate)
                    if canonical:
                        hpo_to_cuis[canonical].add(cui)
                        all_cuis.add(cui)
                        break  # don't double-count

            # --- HGNC: genes ---
            if sab.startswith("HGNC"):
                canonical = hgnc_variant_map.get(code)
                if canonical:
                    hgnc_to_cuis[canonical].add(cui)
                    all_cuis.add(cui)

            # --- RxNorm: drugs ---
            if sab == "RXNORM" and code in rxcui_set:
                rxcui_to_cuis[code].add(cui)
                all_cuis.add(cui)

    return orpha_to_cuis, hpo_to_cuis, hgnc_to_cuis, rxcui_to_cuis, all_cuis


# ---------------------------------------------------------------------------
#  PASS 2: For all CUIs at once, collect English strings
# ---------------------------------------------------------------------------

def collect_english_strings_for_cuis(
    mrconso_path: Path,
    target_cuis: Set[str],
) -> Dict[str, List[str]]:
    """
    Second streaming pass over MRCONSO.RRF.

    For each row where:
        CUI in target_cuis AND LAT == 'ENG',
    collect STR into cui_to_strings[CUI].
    """
    cui_to_strings: Dict[str, List[str]] = {cui: [] for cui in target_cuis}

    if not target_cuis:
        return cui_to_strings

    with mrconso_path.open("r", encoding="utf-8") as f:
        for line in f:
            parts = line.rstrip("\n").split("|")
            if parts and parts[-1] == "":
                parts = parts[:-1]
            if len(parts) < 15:
                continue

            cui = parts[0]      # CUI
            lat = parts[1]      # LAT
            text = parts[14]    # STR

            if cui in target_cuis and lat == "ENG":
                cui_to_strings[cui].append(text)

    return cui_to_strings


# ---------------------------------------------------------------------------
#  Enrichment per entity type
# ---------------------------------------------------------------------------

def enrich_rare_diseases_from_umls(
    conn: sqlite3.Connection,
    orpha_to_cuis: Dict[str, Set[str]],
    cui_to_strings: Dict[str, List[str]],
) -> None:
    """
    For each OrphaCode in orpha_to_cuis:
      - ensure entity('ORPHA:<code>', 'rare_disease') exists
      - insert all normalized MRCONSO English strings for its CUIs into
        rare_disease_name(normalized_name, entity_id).
    """
    cur = conn.cursor()

    for orpha_code, cuis in orpha_to_cuis.items():
        entity_id = f"ORPHA:{orpha_code}"
        insert_entity(conn, entity_id, "rare_disease")

        seen_norm: Set[str] = set()
        for cui in cuis:
            for s in cui_to_strings.get(cui, []):
                norm = normalize_name(s)
                if not norm or norm in seen_norm:
                    continue
                cur.execute(
                    """
                    INSERT OR IGNORE INTO rare_disease_name(normalized_name, entity_id)
                    VALUES (?, ?);
                    """,
                    (norm, entity_id),
                )
                seen_norm.add(norm)

    conn.commit()


def enrich_phenotypes_from_umls(
    conn: sqlite3.Connection,
    hpo_to_cuis: Dict[str, Set[str]],
    cui_to_strings: Dict[str, List[str]],
) -> None:
    """
    For each HPO ID in hpo_to_cuis:
      - ensure entity('<HPO_ID>', 'phenotype') exists
      - insert all normalized MRCONSO English strings for its CUIs into
        phenotype_name(normalized_name, entity_id).
    """
    cur = conn.cursor()

    for hpo_id, cuis in hpo_to_cuis.items():
        entity_id = hpo_id.strip()  # you probably store 'HP:0004322' as-is
        insert_entity(conn, entity_id, "phenotype")

        seen_norm: Set[str] = set()
        for cui in cuis:
            for s in cui_to_strings.get(cui, []):
                norm = normalize_name(s)
                if not norm or norm in seen_norm:
                    continue
                cur.execute(
                    """
                    INSERT OR IGNORE INTO phenotype_name(normalized_name, entity_id)
                    VALUES (?, ?);
                    """,
                    (norm, entity_id),
                )
                seen_norm.add(norm)

    conn.commit()






# ---------------------------------------------------------------------------
#  High-level orchestrator
# ---------------------------------------------------------------------------

def enrich_from_umls(
    conn: sqlite3.Connection,
    mrconso_path: Path,
    orpha_ids: Iterable[str],
    hpo_ids: Iterable[str],
    hgnc_ids: Iterable[str],
    rxcui_ids: Iterable[str],
) -> None:
    """
    High-level function:

      1) Pass 1: build {resource ID -> CUIs} for all resources.
      2) Pass 2: build {CUI -> [English strings]}.
      3) Enrich DB:
           - rare_disease_name   from OrphaID -> CUIs
           - phenotype_name      from HPO ID  -> CUIs
           - gene_name (optional) from HGNC   -> CUIs
           - drug_name (optional) from RxCUI  -> CUIs
    """
    (
        orpha_to_cuis,
        hpo_to_cuis,
        hgnc_to_cuis,
        rxcui_to_cuis,
        all_cuis,
    ) = build_id_to_cui_maps(
        mrconso_path,
        orpha_ids=orpha_ids,
        hpo_ids=hpo_ids,
        hgnc_ids=hgnc_ids,
        rxcui_ids=rxcui_ids,
    )

    cui_to_strings = collect_english_strings_for_cuis(mrconso_path, all_cuis)

    # Enrich per type
    enrich_rare_diseases_from_umls(conn, orpha_to_cuis, cui_to_strings)
    enrich_phenotypes_from_umls(conn, hpo_to_cuis, cui_to_strings)

# No __main__ block on purpose so you can import this cleanly into your project.


In [7]:
from pathlib import Path
from typing import Iterator, Tuple, List

def iter_mrconso_minimal(rrf_path: Path) -> Iterator[Tuple[str, str, str]]:
    """
    Stream MRCONSO.RRF and yield (CUI, LAT, STR) only.

    You *assume* standard RRF order:
      0 CUI | 1 LAT | ... | 14 STR | 15 SRL | 16 SUPPRESS | 17 CVF |
    """
    with rrf_path.open("r", encoding="utf-8") as f:
        for line in f:
            parts = line.rstrip("\n").split("|")
            # trailing '|' → last element is "" → ignore
            if parts and parts[-1] == "":
                parts = parts[:-1]
            if len(parts) < 15:
                continue  # malformed row, skip

            cui = parts[0]
            lat = parts[1]
            text = parts[14]
            yield cui, lat, text


def get_english_strings_for_cui(rrf_path: Path, cui: str) -> List[str]:
    """
    Return all English STR values for a given CUI.
    """
    cui = cui.strip()
    out: List[str] = []
    for row_cui, lat, text in iter_mrconso_minimal(rrf_path):
        if row_cui == cui and lat == "ENG":
            out.append(text)
    return out


if __name__ == "__main__":
    mrconso_path = Path("/home/guests/andreea_magureanu/rare_disease_pipeline/data/raw/MRCONSO.RRF")  

    #cui = "C0431399"
    cui = "C5979921"
    cui = "C0156273"
    names = get_english_strings_for_cui(mrconso_path, cui)
    print(f"{cui}: {len(names)} English strings")
    for s in names:
        print("  -", s)


C0156273: 51 English strings
  - Bladder Diverticulum
  - Bladder Diverticulum
  - diverticulum bladder
  - Diverticulum - bladder
  - Diverticulum - bladder
  - diverticulum; bladder
  - BLADDER DIVERTICULUM
  - BLADDER DIVERTICULUM
  - BLADDER DIVERTICULUM
  - BLADDER, DIVERTICULUM
  - Bladder diverticulum
  - Bladder diverticulum
  - Bladder diverticulum
  - Bladder diverticulum
  - Bladder diverticulum
  - Bladder diverticulum
  - bladder diverticulum
  - bladder; diverticulum
  - Diverticulum of bladder
  - Diverticulum of bladder
  - Diverticulum of bladder
  - Diverticulum of bladder
  - Diverticulum of bladder
  - Diverticulum of bladder
  - Diverticulum of bladder
  - Diverticulum of bladder
  - Diverticulum of bladder
  - Diverticulum of bladder
  - Diverticulum of bladder
  - Diverticulum of bladder
  - Diverticulum of bladder
  - Diverticulum of bladder
  - Diverticulum of bladder
  - Diverticulum of bladder
  - Diverticulum of bladder
  - Diverticulum of bladder NOS
  - Di

In [3]:
import requests
import zipfile
from pathlib import Path

DOWNLOAD_DIR = Path("/home/guests/andreea_magureanu/rare_disease_pipeline/data/raw")

def fetch_file(url: str, dest: Path, params: dict) -> None:
    """
    Simple helper to stream-download a file with requests.
    """
    dest.parent.mkdir(parents=True, exist_ok=True)
    with requests.get(url, params=params, stream=True, timeout=300) as r:
        r.raise_for_status()
        with dest.open("wb") as f:
            for chunk in r.iter_content(chunk_size=8192):
                if chunk:
                    f.write(chunk)


def download_umls_mrsty(api_key: str):
    """
    Download the *current* UMLS Metathesaurus full subset and extract MRSTY.RRF.
    Returns (zip_path, rrf_path).
    """
    base_dir = DOWNLOAD_DIR
    base_dir.mkdir(parents=True, exist_ok=True)

    # 1) Discover current Metathesaurus full subset release
    releases_url = "https://uts-ws.nlm.nih.gov/releases"
    params = {
        "releaseType": "umls-metathesaurus-full-subset",
        "current": "true",
    }
    r = requests.get(releases_url, params=params, timeout=60)
    r.raise_for_status()
    items = r.json()
    if not isinstance(items, list) or not items or "downloadUrl" not in items[0]:
        raise RuntimeError("No UMLS full-subset downloadUrl from UTS Release API.")

    source_url = items[0]["downloadUrl"]   # big umls-YYYYAA-metathesaurus-subset.zip
    zip_path = base_dir / Path(source_url).name

    # 2) Download via UTS Download API
    dl_api = "https://uts-ws.nlm.nih.gov/download"
    fetch_file(dl_api, zip_path, params={"url": source_url, "apiKey": api_key})

    # 3) Extract MRSTY.RRF
    rrf_path = None
    with zipfile.ZipFile(zip_path, "r") as zf:
        for name in zf.namelist():
            # MRSTY.RRF usually lives in the META/ or root; be tolerant
            if name.endswith("MRSTY.RRF"):
                rrf_path = base_dir / "MRSTY.RRF"
                with zf.open(name) as src, open(rrf_path, "wb") as dst:
                    dst.write(src.read())
                break

    if rrf_path is None:
        raise RuntimeError("MRSTY.RRF not found inside the downloaded UMLS zip.")

    return zip_path, rrf_path


In [4]:
download_umls_mrsty(API_KEY)

(PosixPath('/home/guests/andreea_magureanu/rare_disease_pipeline/data/raw/umls-2025AB-metathesaurus-full.zip'),
 PosixPath('/home/guests/andreea_magureanu/rare_disease_pipeline/data/raw/MRSTY.RRF'))

In [ ]:
from pathlib import Path
from collections import defaultdict
from typing import Dict, Set, List


def build_orphaid_to_cui_index(mrconso_path: Path) -> Dict[str, Set[str]]:
    """
    One pass over MRCONSO.RRF and build:
        { orpha_id (CODE) -> {CUI1, CUI2, ...} }

    Assumes standard MRCONSO field order:
      0 CUI | 1 LAT | ... | 11 SAB | 12 TTY | 13 CODE | 14 STR | ...
    and that Orphanet rows have SAB starting with 'ORPHANET'.
    """
    index: Dict[str, Set[str]] = defaultdict(set)

    with mrconso_path.open("r", encoding="utf-8") as f:
        for line in f:
            parts = line.rstrip("\n").split("|")
            if parts and parts[-1] == "":
                parts = parts[:-1]
            if len(parts) < 14:
                continue

            cui  = parts[0]   # CUI
            sab  = parts[11]  # SAB (source)
            code = parts[13]  # CODE (for Orphanet: OrphaCode)

            # Keep only Orphanet rows with a valid OrphaCode
            if sab.startswith("ORPHANET") and code:
                # store as string; Orphanet codes are typically like "558"
                index[code].add(cui)
           


    return index


def get_cuis_for_orphaid(
    orphaid_to_cui: Dict[str, Set[str]],
    orpha_id: str | int,
) -> List[str]:
    """
    Lookup helper: given an OrphaCode, return all CUIs that reference it.
    """
    key = str(orpha_id).strip()
    cuis = orphaid_to_cui.get(key, set())
    return sorted(cuis)


if __name__ == "__main__":
    mrconso_path = Path("/home/guests/andreea_magureanu/rare_disease_pipeline/data/raw/MRCONSO.RRF")  # adjust path

    # 1) Build the OrphaID -> CUI index once
    orpha_to_cui = build_orphaid_to_cui_index(mrconso_path)

    # 2) Query for a specific OrphaID
    orpha_id = "475"
    # hpo = "0000015"
    cuis = get_cuis_for_orphaid(orpha_to_cui, orpha_id)
    print(f"OrphaID {orpha_id} has {len(cuis)} CUIs: {cuis}")


OrphaID 475 has 0 CUIs: []


In [3]:
downloaders.download_umls_mrconso(API_KEY)

(PosixPath('/home/guests/andreea_magureanu/rare_disease_pipeline/data/raw/umls-2025AB-mrconso.zip'),
 PosixPath('/home/guests/andreea_magureanu/rare_disease_pipeline/data/raw/MRCONSO.RRF'))